In [1]:
%matplotlib widget
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy
import h5py
import os
print(os.getpid())
%cd ../../

import pylib.mix as mix
import cvxpy as cp
import pylib.qucf_read as qucf_r
import pylib.measurement as mse

import pylib.Chebyschev_coefs as ch

from matplotlib import colors
colors_ = ["blue", "red", "green", "gray", "black"]

plt.rcParams.update({
    "text.usetex": True,
    'text.latex.preamble': r"\usepackage{amsmath} \boldmath"
})

14627
/media/work/docs/codes/QuCF/scripts-py


In [2]:
path_save_ = "./jupyter-notebooks/Stepanoff/results/"
path_qucf_ = "../QuCF/simulations/Stepanoff/init_mod"
for i in range(100):
    plt.close()

In [67]:
# -----------------------------------------------------
def plot_2D_subplot(
        ax, fig, X, Y, f, 
        str_title="", 
        fontsize = 30, 
        label_x = '$\\theta\'/\pi$', label_y = "$\phi\'/\pi$",
        flag_x = True, flag_y = True
):
    f_max_loc = np.max(np.abs(f))
    f_min_loc = np.min(f)

    cmap = plt.get_cmap("seismic")
    levels = np.linspace(-f_max_loc, f_max_loc, 101) 
    # levels = np.linspace(f_min_loc, f_max_loc, 101) 
    divnorm = colors.BoundaryNorm(levels, ncolors=cmap.N, clip=True)

    cs = ax.pcolormesh(X, Y, f, cmap=cmap, norm=divnorm, shading='gouraud') 
    cb = fig.colorbar(cs, ax = ax)
    cb.ax.ticklabel_format(style="scientific")
    cb.ax.tick_params(labelsize=fontsize)

    offset_text = cb.ax.yaxis.get_offset_text()
    offset_text.set_fontsize(fontsize) 
    offset_text.set_x(4) 

    ax.set_xlabel(label_x, fontsize = fontsize)
    ax.set_ylabel(label_y, fontsize = fontsize)
    ax.set_title(str_title, fontsize = fontsize)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    # ax.grid()

    if not flag_x:
        ax.set_xticklabels([])
    if not flag_y:
        ax.set_yticklabels([])

    return
# -----------------------------------------------------
def create_diag_matrix(t, N_half, alpha, name_matrix, flag_cut):
    if not flag_cut:
        N = 2 * N_half + 1
        c_arr = np.arange(-N_half, N_half+1)
    else:
        N = 2 * N_half
        c_arr = np.arange(-N_half, N_half)

    aa = np.linspace(0, 2.*np.pi, N)
    diag = np.zeros(N**2)
    count = -1

    if name_matrix == "M1":
        for i2 in range(N):
            for i1 in range(N):
                count += 1
                diag[count] = - (1. - alpha) * c_arr[i1] * (1. - np.cos(aa[i2]))

    if name_matrix == "M2":
        for i2 in range(N):
            for i1 in range(N):
                count += 1
                diag[count] = - alpha * c_arr[i2] * (1. - np.cos(aa[i1]))

    # --- reference unitary matrix ---
    Nsq = N**2
    diag_U_ref = np.zeros(Nsq, dtype = complex)
    for ii in range(Nsq):
        diag_U_ref[ii] = np.exp(-1j * t * diag[ii])
    return diag_U_ref
# -----------------------------------------------------
def f_init_1d_(x):
    f_loc = lambda x1: np.exp(kappa_ * np.sin(x1/2.)**2) - 1.
    y = f_loc(x) / f_loc(np.pi)
    return y
# -----------------------------------------------------
def f_init_2d_(x, Nx):
    g_cl = f_init_1d_(x)

    y_2D = np.zeros((Nx, Nx))
    f_loc = lambda x1, x2: np.exp(kappa_ * np.sin((x1 + x2)/2.)**2) - 1
    for i_phi in range(Nx):
        for i_theta in range(Nx):
            y_2D[i_theta, i_phi] = f_loc(x[i_theta], x[i_phi]) * g_cl[i_phi]
    y_2D /= f_loc(np.pi, 0.)
    return y_2D
# -----------------------------------------------------
flag_cut_ = True

n_half_, kappa_ = 9, 1.

dt_, Nt_ = 0.001, 1000   # total time interval is dt_ * Nt_
# dt_, Nt_ = 0.01, 200   # total time interval is dt_ * Nt_

alpha_ = np.sqrt(20.)

# --- spatial grid ---
n_ = n_half_ + 1
N_half_ = 1 << n_half_
N_ = 2 * N_half_ + int(not flag_cut_)
x_ = np.linspace(0., 2.*np.pi, N_)

# --- initial conditions ---
init_2d_ = f_init_2d_(x_, N_)

# --- unitary operators ---
diag_U_M1_  = create_diag_matrix(dt_, N_half_, alpha_, "M1", flag_cut_)
diag_U_M2_  = create_diag_matrix(dt_, N_half_, alpha_, "M2", flag_cut_)

In [68]:
# -------------------------------------------------------------------
# --- Classical simulations in real space  ---
# -------------------------------------------------------------------
def form_file_name(alpha, kappa, nx, prefix = ""):
    file_name = "{:s}Stepanoff_data_k{:0.1f}_a{:0.3f}_n{:d}.hdf5".format(prefix, kappa, alpha, nx)
    return file_name
# -----------------------------------------------------------------------------------
def load_data(full_name):
    print("Reading the coefficients from:\n " + full_name)
    with h5py.File(full_name, "r") as f:
        grp = f["basic"]

        nx = int(grp["nx"][()])
        N_data = int(grp["N_data"][()])
        alpha = grp["alpha"][()]
        kappa = grp["kappa"][()]
        theta = np.array(grp["theta"])

        fx_arr = []
        t_arr  = []
        for id_data in range(N_data):
            grp = f["data_{:d}".format(id_data)]
            fx = np.array(grp["fx"])
            t  = float(grp["t"][()])
            fx_arr.append(fx)
            t_arr.append(t)

    # --- Form grids ---
    Nx = 1 << nx
    x1, x2 = np.meshgrid(theta, theta)
    x = np.concatenate((x1[:, :, np.newaxis], x2[:, :, np.newaxis]), axis=2)
    x1 = x1 / np.pi
    x2 = x2 / np.pi
    x = np.reshape(x, (Nx ** 2, 2))

    return fx_arr, t_arr, Nx, nx, alpha, kappa, x1, x2, x
# -----------------------------------------------------------------------------------
def get_Dopri_classical_simulations(prefix = ""):
    full_name = form_file_name(alpha_, kappa_, n_, prefix)

    print("--- Looking for the file: {:s} in the path {:s}".format(full_name, path_save_))
    full_name = os.path.join(path_save_, full_name)
    if os.path.isfile(full_name):
        print("--- Loading data ---")
        y_cl_dopri_arr, t_arr, _, _, _, _, _, _, _, = load_data(full_name)
    else:
        print("file is not found")

    # --- Find a necessary simulation ---
    t_ref = int(Nt_ * dt_)
    t_arr_ = np.array(t_arr, dtype = int)
    pos_arr = np.where(t_arr_ == t_ref)[0][0]
    print()
    print("Take the Dopri simulation at t = {:d}".format(t_arr_[pos_arr]))
    y_cl_dopri_1d = y_cl_dopri_arr[pos_arr]

    # --- Normalize to 1 ---
    y_cl_dopri_1d /= np.max(np.abs(y_cl_dopri_1d))

    # --- from 1D to 2D ---
    # y_cl_dopri_2d = rearrange_into_2d(y_cl_dopri_1d, Nx_)
    y_cl_dopri_2d = np.reshape(y_cl_dopri_1d, (N_, N_)) 

    # # --- Plotting ---
    # X, Y = np.meshgrid(x_/np.pi, x_/np.pi)
    # fig1, axs = plt.subplots(1, 1, figsize=(10,8))
    # plot_2D_subplot(
    #     axs, fig1, X, Y, y_cl_dopri_2d,  
    #     str_title="CL-real: $" + "t={:0.2f}".format(Nt_*dt_) + "$", 
    #     label_x = '$\\theta/\pi$', label_y = "$\phi/\pi$"
    # )
    # plt.tight_layout()

    return y_cl_dopri_2d
# -----------------------------------------------------------------------------------
def compare_dopri(y_orig, y_new):
    # --- Error ---
    err_cl = y_orig - y_new
    max_abs_err_cl = np.max(np.abs(err_cl))
    print("max. abs. err : {:0.3e}".format(max_abs_err_cl))

    # --- Plotting ---
    X, Y = np.meshgrid(x_/np.pi, x_/np.pi)
    fig1, axs = plt.subplots(3, 1, figsize=(10,18))
    plot_2D_subplot(
        axs[0], fig1, X, Y, y_orig,  
        str_title="$" + "t={:0.2f}".format(Nt_*dt_) + "$", 
        label_x = '', label_y = "$\phi/\pi$", 
        flag_x = False
    )
    plot_2D_subplot(
        axs[1], fig1, X, Y, y_new,  
        label_x = '', label_y = "$\phi/\pi$", 
        flag_x = False
    )
    plot_2D_subplot(
        axs[2], fig1, X, Y, err_cl,   
        label_x = '$\\theta/\pi$', label_y = "$\phi/\pi$"
    )
    plt.tight_layout()
    return
# -------------------------------------------------------------------
# y_cl_dopri_ = get_Dopri_classical_simulations()
y_cl_dopri_new_ = get_Dopri_classical_simulations("new_")

# compare_dopri(y_cl_dopri_, y_cl_dopri_new_)

--- Looking for the file: new_Stepanoff_data_k1.0_a4.472_n10.hdf5 in the path ./jupyter-notebooks/Stepanoff/results/
--- Loading data ---
Reading the coefficients from:
 ./jupyter-notebooks/Stepanoff/results/new_Stepanoff_data_k1.0_a4.472_n10.hdf5

Take the Dopri simulation at t = 1


In [69]:
# -----------------------------------------------------------------
# --- Computation ---
# -----------------------------------------------------------------
def computation_in_real_and_fourier(diag_U_M1, diag_U_M2, flag_plot):
    # ----------------------------------------------------------
    def rearrange_into_2d(y, N):
        y_new = np.zeros((N, N), dtype = y.dtype)
        for ic in range(N):
            for ir in range(N):
                y_new[ir, ic] = y[ir + ic * N]
        return y_new
    # ----------------------------------------------------------
    def evolution_one_matrix(psi_init_x, id_axis):
        init_fft_2d = np.fft.fft(psi_init_x, axis = id_axis)
        init_fft_2d = np.fft.fftshift(init_fft_2d, axes = id_axis)
        init_fft_1d = init_fft_2d.flatten(order = "F")  # CHECK; 

        if id_axis == 0:
            psi_t_fourier_1D = diag_U_M1 * init_fft_1d
        else:
            psi_t_fourier_1D = diag_U_M2 * init_fft_1d

        psi_t_fourier_2D = rearrange_into_2d(psi_t_fourier_1D, N_)
        psi_t_fourier_2D = np.fft.ifftshift(psi_t_fourier_2D, axes = id_axis)
        psi_t_x = np.fft.ifft(psi_t_fourier_2D, axis = id_axis)
        return psi_t_x
    # ----------------------------------------------------------

    # --- Computation ---
    psi_t_x = np.conjugate(np.array(init_2d_))
    # psi_t_x = np.array(init_2d_)
    for ii_t in range(Nt_):
        # --- evolution in Fourier over theta ---
        psi_t_x = evolution_one_matrix(psi_t_x, 0)

        # --- evolution in Fourier over phi ---
        psi_t_x = evolution_one_matrix(psi_t_x, 1)

    # --- Plotting ---
    X, Y = np.meshgrid(x_/np.pi, x_/np.pi)
    X_new_prel = X + Y
    X_new = np.where(
        X_new_prel <= 2., X_new_prel, 2. - X_new_prel
    )

    y_orig = np.array(psi_t_x.real.transpose())
    y_new  = np.zeros((N_, N_))
    for ir in range(N_):
        for ic in range(N_):
            ic_new = ic + ir
            if ic_new >= N_:
                ic_new = ic_new - N_
            y_new[ir, ic_new] = y_orig[ir, ic]

    y_new /= np.max(np.abs(y_new))

    # --- Plotting ---
    if flag_plot:
        fig1, axs = plt.subplots(1, 1, figsize=(10,8))
        plot_2D_subplot(
            axs, fig1, X, Y, y_new,  
            str_title="$" + "t={:0.2f}".format(Nt_*dt_) + "$", 
            label_x = '$\\theta/\pi$', label_y = "$\phi/\pi$"
        )
        plt.tight_layout()
    return y_new
# ------------------------------------------------------------------------
y_cl_mix_ = computation_in_real_and_fourier(diag_U_M1_, diag_U_M2_, flag_plot = False)

In [70]:
# -------------------------------------------------------------------
# --- Compare classical simulations in real and mix spaces ---
# -------------------------------------------------------------------
def compare_classical(y_cl_mix, y_cl_dopri, flag_plot = False):
    # --- Error ---
    err_cl = y_cl_dopri - y_cl_mix
    max_abs_err_cl = np.max(np.abs(err_cl))
    print("max. abs. err : {:0.3e}".format(max_abs_err_cl))

    # --- Plotting ---
    if flag_plot:
        X, Y = np.meshgrid(x_/np.pi, x_/np.pi)
        fig1, axs = plt.subplots(3, 1, figsize=(10,18))
        plot_2D_subplot(
            axs[0], fig1, X, Y, y_cl_mix,  
            str_title="$" + "t={:0.2f}".format(Nt_*dt_) + "$", 
            label_x = '', label_y = "$\phi/\pi$", 
            flag_x = False
        )
        plot_2D_subplot(
            axs[1], fig1, X, Y, y_cl_dopri,  
            label_x = '', label_y = "$\phi/\pi$", 
            flag_x = False
        )
        plot_2D_subplot(
            axs[2], fig1, X, Y, err_cl,   
            label_x = '$\\theta/\pi$', label_y = "$\phi/\pi$"
        )
        plt.tight_layout()
    return
# -------------------------------------------------------------------
compare_classical(y_cl_mix_, y_cl_dopri_new_, flag_plot = False)

max. abs. err : 1.605e-02


In [ ]:
def scan_dt():
    nx, t = 10, 1.
    dt_array  = [      0.1,      0.01,     0.001]
    err_array = [4.644e-01, 6.706e-02, 1.605e-02]
    return
# --------------------------------------------------------------
scan_dt()